# PFAS Groundwater — T1 Binary Baseline (Colab)

**Task T1a**: predict whether a well exceeds the regulatory PFAS threshold (binary).

This notebook is **AUTONOMOUS**: it bootstraps `src/`, downloads the dataset from Drive,
and runs the full baseline pipeline (LR / RF / XGBoost, spatial + random CV, SHAP, ablations).

**Strict predictive mode**: no PFAS measurement is used as a feature (CLAUDE.md §1).

---
**PARAMETERS TO SET (cell below)**:
- `SMOKE_TEST` : `True` for a fast CPU sanity check (~3 min), `False` for the full GPU run.
- `BOOTSTRAP` : `"git"` to clone from GitHub, `"drive"` to copy `src/` from your Drive.
- `REPO_URL` / `GIT_REF` : GitHub repo URL and branch/commit (used when `BOOTSTRAP="git"`).
- `DRIVE_PROJECT_DIR` : absolute path on your mounted Drive (used for `BOOTSTRAP="drive"` and for saving artifacts).
- `DRIVE_DATA_PATH` : path on mounted Drive to `CA-PFAS-ASGWS.parquet`, **OR** set `GDRIVE_FILE_ID` to use `gdown`.
- `TARGET` : `"T1a"` (default) or `"T1b"`.

In [ ]:
# ============================================================
#  USER PARAMETERS — edit these before running
# ============================================================

SMOKE_TEST = True          # True = fast CPU sanity check; False = full GPU run

# --- Code bootstrap mode ---
# "git"   -> git clone REPO_URL@GIT_REF (needs up-to-date code on remote)
# "drive" -> copy src/ from DRIVE_PROJECT_DIR/src/
BOOTSTRAP = "git"
REPO_URL  = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF   = "main"   # branch name or full commit SHA

# --- Drive paths (used for both bootstrap='drive' and artifact saving) ---
# Mount path on Colab: usually /content/drive/MyDrive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/pfas-gnn"   # <-- SET THIS

# --- Dataset (parquet file) ---
# Option A: path on mounted Drive
DRIVE_DATA_PATH = "/content/drive/MyDrive/pfas-gnn/data/CA-PFAS-ASGWS.parquet"  # <-- SET THIS
# Option B: gdown (set GDRIVE_FILE_ID to non-empty string to use gdown instead)
GDRIVE_FILE_ID = ""   # e.g. "1AbCdEfGh..." from the Google Drive share URL

# --- Run target ---
TARGET = "T1a"   # "T1a" (regulatory) or "T1b" (sum > 70 ng/L)

print(f"SMOKE_TEST={SMOKE_TEST}  BOOTSTRAP={BOOTSTRAP!r}  TARGET={TARGET}")

## Cell 1 — GPU detection & version info

In [ ]:
import sys, platform
print(f"Python : {sys.version}")
print(f"Platform: {platform.platform()}")

# GPU check (tolerant outside Colab)
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    import subprocess
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if r.returncode == 0:
        print("\n[GPU detected]")
        print(r.stdout[:800])
    else:
        print("[WARNING] No GPU found — set Runtime > Change runtime type > GPU in Colab.")
        print(r.stderr[:200])
else:
    print("[Local run — no GPU check performed]")

# Library versions (no torch needed for these baselines)
try:
    import numpy as np; print(f"numpy   : {np.__version__}")
except ImportError: print("numpy   : NOT INSTALLED")
try:
    import sklearn; print(f"sklearn : {sklearn.__version__}")
except ImportError: print("sklearn : NOT INSTALLED")
try:
    import xgboost as xgb; print(f"xgboost : {xgb.__version__}")
except ImportError: print("xgboost : not installed (HGB fallback will be used)")
try:
    import pandas as pd; print(f"pandas  : {pd.__version__}")
except ImportError: print("pandas  : NOT INSTALLED")

## Cell 2 — Install dependencies (Colab only)

In [ ]:
if IN_COLAB:
    import subprocess, sys
    pkgs = [
        "xgboost>=2.0,<3.0",
        "optuna>=3.5,<4.0",
        "shap>=0.44,<1.0",
        "imbalanced-learn>=0.11,<1.0",
        "pyyaml>=6.0,<7.0",
        "pandas>=2.0,<3.0",
        "scikit-learn>=1.4,<2.0",
        "pyarrow>=14.0",
        "scipy>=1.11",
    ]
    print("Installing packages...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr[-1000:])
        raise RuntimeError("Dependency installation failed.")
    print("Packages installed. Verifying imports...")
    import xgboost, optuna, shap, imblearn, yaml, pandas, sklearn
    print(f"  xgboost={xgboost.__version__}  optuna={optuna.__version__}  "
          f"shap={shap.__version__}  imblearn={imblearn.__version__}")
    print("Imports OK.")
else:
    print("[Local run] Skipping pip install — using local environment.")

## Cell 3 — Bootstrap `src/` code

**IMPORTANT — code must be up-to-date on the remote (git) or on Drive (drive)**.

If `BOOTSTRAP="git"`: the remote must contain `src/baselines_t1.py`, `src/baselines_t2.py`,
`src/metrics.py` (with `FrequencyClassChain` and the 5 required metrics). If those files
are not yet pushed, push them first or use `BOOTSTRAP="drive"` with a local copy.

If `BOOTSTRAP="drive"`: copy `pfas-gnn/src/` to your Drive at `DRIVE_PROJECT_DIR/src/`.

In [ ]:
import sys, shutil
from pathlib import Path

REPO_LOCAL = Path("/content/pfas-gnn") if IN_COLAB else Path("..").resolve()

if IN_COLAB:
    if BOOTSTRAP == "git":
        import subprocess
        if REPO_LOCAL.exists():
            print(f"Repo already cloned at {REPO_LOCAL} — pulling latest...")
            r = subprocess.run(["git", "-C", str(REPO_LOCAL), "fetch", "origin"],
                               capture_output=True, text=True)
            r2 = subprocess.run(["git", "-C", str(REPO_LOCAL), "checkout", GIT_REF],
                                capture_output=True, text=True)
            r3 = subprocess.run(["git", "-C", str(REPO_LOCAL), "reset", "--hard",
                                 f"origin/{GIT_REF}"],
                                capture_output=True, text=True)
            print(r3.stdout or r3.stderr)
        else:
            print(f"Cloning {REPO_URL} @ {GIT_REF} ...")
            r = subprocess.run(
                ["git", "clone", "--depth=1", "--branch", GIT_REF, REPO_URL, str(REPO_LOCAL)],
                capture_output=True, text=True
            )
            if r.returncode != 0:
                print("[ERROR] git clone failed:")
                print(r.stderr)
                raise RuntimeError("git clone failed. Check REPO_URL / GIT_REF.")
            print("Cloned OK.")
    elif BOOTSTRAP == "drive":
        src_drive = Path(DRIVE_PROJECT_DIR) / "src"
        dst_src   = REPO_LOCAL / "src"
        if not src_drive.exists():
            raise FileNotFoundError(
                f"src/ not found on Drive at {src_drive}. "
                "Copy pfas-gnn/src/ to your Drive first."
            )
        if dst_src.exists():
            shutil.rmtree(dst_src)
        shutil.copytree(str(src_drive), str(dst_src))
        print(f"Copied src/ from Drive: {src_drive} -> {dst_src}")
    else:
        raise ValueError(f"Unknown BOOTSTRAP mode: {BOOTSTRAP!r}. Use 'git' or 'drive'.")
else:
    # Local run: repo root is parent of notebooks/
    REPO_LOCAL = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path("..").resolve()
    print(f"[Local run] Using repo at {REPO_LOCAL}")

# Add repo root to sys.path so `from src.xxx import yyy` works
repo_str = str(REPO_LOCAL)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
print(f"sys.path[0] = {sys.path[0]}")

In [ ]:
# --- Guard-rail: verify the code is up-to-date ---
import importlib, sys

# Force reimport in case src was already imported with old path
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

try:
    import src.baselines_t1
    import src.baselines_t2
    import src.metrics
except ImportError as e:
    raise ImportError(
        f"CRITICAL: Cannot import src modules: {e}\n"
        "If BOOTSTRAP='git': push baselines_t1.py, baselines_t2.py, metrics.py to the remote first.\n"
        "If BOOTSTRAP='drive': ensure src/ on Drive is up-to-date."
    )

# Check for FrequencyClassChain (proves code is not obsolete)
assert hasattr(src.baselines_t2, "FrequencyClassChain"), (
    "CRITICAL: FrequencyClassChain not found in src.baselines_t2.\n"
    "The code on Colab is OBSOLETE.\n"
    "Action: push the latest baselines_t2.py to the remote (git) OR "
    "copy src/ with the up-to-date files to your Drive (drive)."
)

# Check metrics module exports the 5 required metrics
from src.metrics import binary_metrics, multilabel_metrics, REQUIRED
assert set(REQUIRED) == {"roc_auc", "f1", "accuracy", "recall", "precision"}, (
    "CRITICAL: src.metrics.REQUIRED does not match the 5 required metrics. Code is obsolete."
)

print("[guard-rail] src/ is up-to-date: FrequencyClassChain found, metrics OK.")
from src.baselines_t1 import run_baselines
print("[guard-rail] run_baselines imported successfully.")

## Cell 4 — Mount Drive & load dataset

In [ ]:
import os
from pathlib import Path

DATA_PATH = None

if IN_COLAB:
    # Mount Google Drive (will prompt for auth if not already mounted)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted.")

    if GDRIVE_FILE_ID:
        # Option B: gdown
        import subprocess
        dest = Path("/content/CA-PFAS-ASGWS.parquet")
        if not dest.exists():
            print(f"Downloading dataset via gdown (ID={GDRIVE_FILE_ID})...")
            r = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "gdown"],
                capture_output=True, text=True
            )
            import gdown
            gdown.download(id=GDRIVE_FILE_ID, output=str(dest), quiet=False)
        else:
            print(f"Dataset already at {dest}.")
        DATA_PATH = dest
    else:
        # Option A: mounted Drive path
        DATA_PATH = Path(DRIVE_DATA_PATH)
        if not DATA_PATH.exists():
            raise FileNotFoundError(
                f"Dataset not found at {DATA_PATH}.\n"
                "Set DRIVE_DATA_PATH to the parquet file on your mounted Drive, "
                "or set GDRIVE_FILE_ID to download via gdown."
            )
        print(f"Dataset path: {DATA_PATH}  size={DATA_PATH.stat().st_size/1e6:.1f} MB")
else:
    # Local run: use local data file
    DATA_PATH = REPO_LOCAL / "data" / "CA-PFAS-ASGWS.parquet"
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Local dataset not found at {DATA_PATH}. "
            "For local runs, ensure data/CA-PFAS-ASGWS.parquet exists."
        )
    print(f"[Local] Dataset: {DATA_PATH}  size={DATA_PATH.stat().st_size/1e6:.1f} MB")

# Integrity check
import pandas as pd
print("Loading parquet for integrity check...")
_df_check = pd.read_parquet(DATA_PATH)
n_rows, n_cols = _df_check.shape
print(f"Shape: {n_rows} x {n_cols}")

REQUIRED_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL"]
missing_cols = [c for c in REQUIRED_COLS if c not in _df_check.columns]
if missing_cols:
    raise ValueError(
        f"INTEGRITY FAIL: columns {missing_cols} missing from the parquet.\n"
        "Check that DRIVE_DATA_PATH / GDRIVE_FILE_ID points to the correct file."
    )
if n_cols != 201:
    print(f"[WARNING] Expected 201 columns, got {n_cols}. Check dataset version.")
if n_rows != 46338:
    print(f"[WARNING] Expected 46338 rows, got {n_rows}. This may be an older snapshot.")
if n_cols == 201 and n_rows == 46338:
    print("[INTEGRITY OK] shape==(46338, 201), required columns present.")
del _df_check

# Point src/config.py DATA_PARQUET at the resolved path
import src.config as C
C.DATA_PARQUET = DATA_PATH
print(f"src.config.DATA_PARQUET -> {C.DATA_PARQUET}")

## Cell 5 — Experiment output directory

In [ ]:
from pathlib import Path
import time

if IN_COLAB:
    SAVE_DIR = Path(DRIVE_PROJECT_DIR) / "experiments" / f"baseline_t1{'_smoke' if SMOKE_TEST else ''}"
else:
    SAVE_DIR = REPO_LOCAL / "experiments" / f"baseline_t1{'_smoke' if SMOKE_TEST else ''}"

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {SAVE_DIR}")

## Cell 6 — Run baseline T1

**Smoke test** (`SMOKE_TEST=True`): ~500 wells, 3 spatial folds, 3 Optuna trials, CPU < 3 min.

**Full run** (`SMOKE_TEST=False`): all ~11 333 wells, 8 spatial + 8 random folds, 20 Optuna trials.
Estimated duration on CPU: **~45–90 min**. On Colab GPU (sklearn/XGBoost use CPU cores anyway),
expect **~20–45 min** depending on Colab instance.

> Note: scikit-learn and XGBoost use CPUs (`n_jobs=-1`). A Colab High-RAM CPU instance
> may outperform a GPU instance for this pure-sklearn baseline. Select runtime accordingly.

In [ ]:
import logging, time
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True
)

t_start = time.time()
print(f"{'='*60}")
print(f"T1 BASELINE RUN")
print(f"  target     = {TARGET}")
print(f"  smoke_test = {SMOKE_TEST}")
print(f"  save_dir   = {SAVE_DIR}")
print(f"{'='*60}")

results = run_baselines(
    smoke=SMOKE_TEST,
    target=TARGET,
    run_ablations_flag=True,
    run_shap_flag=True,
    save_dir=SAVE_DIR,
)

elapsed = time.time() - t_start
print(f"\nRun completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

## Cell 7 — Results: models × 5 metrics (spatial CV + random CV + Δ)

In [ ]:
import pandas as pd
import numpy as np

print("\n" + "="*80)
print(f"MODELS x 5 METRICS — Target={TARGET}  smoke={SMOKE_TEST}")
print("="*80)

rows = []
for mn, res in results["model_results"].items():
    sp, rd, dlt = res["spatial"], res["random"], res["delta"]
    g = lambda d, k: d.get(f"{k}_mean", float("nan"))
    rows.append({
        "model":        mn,
        # the 5 required metrics — spatial CV (reference)
        "AUC-ROC (sp)": g(sp, "roc_auc"),
        "F1 (sp)":      g(sp, "f1"),
        "Accuracy (sp)": g(sp, "accuracy"),
        "Recall (sp)":  g(sp, "recall"),
        "Precision (sp)": g(sp, "precision"),
        # random CV
        "AUC-ROC (rd)": g(rd, "roc_auc"),
        "F1 (rd)":      g(rd, "f1"),
        # spatial artefact
        "Δ AUC-ROC":   dlt.get("roc_auc", float("nan")),
        "Δ F1":        dlt.get("f1", float("nan")),
        "Global OOF AUC": res.get("global_auc_spatial_oof", float("nan")),
    })

df_res = pd.DataFrame(rows).set_index("model")
pd.options.display.float_format = "{:.3f}".format
print(df_res.to_string())

print("\nInterpretation:")
print("  Δ AUC-ROC = random - spatial (positive -> spatial autocorrelation artefact).")
print("  Spatial CV is the reference; random CV inflates scores when wells are autocorrelated.")

## Cell 8 — Paired comparisons (Nadeau-Bengio + Wilcoxon)

In [ ]:
print("\n" + "="*60)
print("PAIRED COMPARISONS (spatial folds)")
print("="*60)

for pair, comp in results["comparison"].items():
    nb = comp["nadeau_bengio"]
    wc = comp["wilcoxon"]
    a, b = pair.split("_vs_")
    sig_nb = "*" if (not np.isnan(nb["p"]) and nb["p"] < 0.05) else ""
    sig_wc = "*" if (not np.isnan(wc["p"]) and wc["p"] < 0.05) else ""
    print(f"  {a} vs {b}:")
    print(f"    mean ΔAUC = {nb['mean_diff']:+.4f}")
    print(f"    Nadeau-Bengio corrected t-test: p = {nb['p']:.4f}{sig_nb}")
    print(f"    Wilcoxon signed-rank:           p = {wc['p']:.4f}{sig_wc}")
    print(f"    (noise threshold = {comp.get('noise_threshold_auc', 0.03)})")
    print()

## Cell 9 — Top SHAP / feature importances

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

imp = results["importance"]
if not imp.empty:
    top_n = min(20, len(imp))
    top = imp.head(top_n)
    method = top["method"].iloc[0]
    print(f"\nTop {top_n} features  [{method}]:\n")
    for _, row in top.iterrows():
        bar = "#" * int(row["importance"] / (imp["importance"].max() + 1e-9) * 30)
        print(f"  {row['feature']:<40} {row['importance']:.4f}  {bar}")

    # Optional bar plot
    try:
        fig, ax = plt.subplots(figsize=(8, top_n * 0.35 + 1))
        ax.barh(top["feature"][::-1], top["importance"][::-1], color="steelblue")
        ax.set_xlabel(f"Importance ({method})")
        ax.set_title(f"Top {top_n} features — T1 baseline")
        plt.tight_layout()
        fig_path = SAVE_DIR / "feature_importance.png"
        plt.savefig(fig_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Figure saved to {fig_path}")
    except Exception as e:
        print(f"[plot skipped: {e}]")
else:
    print("No feature importance computed (SHAP/permutation was disabled or failed).")

## Cell 10 — Ablations (feature configuration)

In [ ]:
abl = results["ablations"]
if abl:
    print("\n" + "="*60)
    print("ABLATIONS — RF, no Optuna (speed), spatial + random CV")
    print("="*60)
    abl_rows = []
    for key, v in abl.items():
        abl_rows.append({
            "config":        key,
            "n_features":    v["n_features"],
            "AUC sp":        v["spatial_roc_auc"],
            "AUC rd":        v["random_roc_auc"],
            "Δ AUC":         v["delta_roc_auc"],
            "Recall sp":     v["spatial_recall"],
        })
    df_abl = pd.DataFrame(abl_rows).set_index("config")
    print(df_abl.to_string(float_format="{:.3f}".format))
    print()
    print("Ablation configurations:")
    print("  no_loc_all   — all features except lat/lon (reference)")
    print("  with_loc_all — include lat/lon as features (fuite spatiale risk)")
    print("  no_loc_core  — core hydrogeochemical cocontaminants only")
    print("  no_loc_none  — no cocontaminants")
else:
    print("Ablations not run (run_ablations_flag was False).")

## Cell 11 — Spatial artefact analysis

In [ ]:
bp = results["block_prevalence"]
print("\n" + "="*60)
print("SPATIAL BLOCKS — prevalence per block")
print("="*60)
print(f"  {len(bp)} blocks, prevalence range: "
      f"[{bp['prevalence'].min():.3f}, {bp['prevalence'].max():.3f}]")
print(f"  std(prevalence) = {bp['prevalence'].std():.3f}")
print()
print("  Block-level prevalence variation indicates geographic heterogeneity.")
print("  High variation -> spatial CV is essential to avoid overoptimistic scores.")
print(bp.to_string(index=False))

## Cell 12 — Save summary & timing

In [ ]:
import json

print(f"\nArtefacts written to: {SAVE_DIR}")
for f in sorted(SAVE_DIR.iterdir()):
    print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

# Timing summary + extrapolation
from src.baselines_t1 import SMOKE_N_WELLS, SMOKE_OUTER_K, OPTUNA_TRIALS_SMOKE, OUTER_SPATIAL_K, OPTUNA_TRIALS_FULL
print(f"\nRun duration: {elapsed:.1f}s ({elapsed/60:.2f} min)")
if SMOKE_TEST:
    n_full = 11333
    scale = (n_full / SMOKE_N_WELLS) * (OUTER_SPATIAL_K / SMOKE_OUTER_K) * (OPTUNA_TRIALS_FULL / OPTUNA_TRIALS_SMOKE)
    est_min = elapsed * scale / 60
    print(f"\nEXTRAPOLATED full run: ~{est_min:.0f} min on CPU")
    print(f"  (x{n_full/SMOKE_N_WELLS:.0f} wells, x{OUTER_SPATIAL_K/SMOKE_OUTER_K:.0f} folds, x{OPTUNA_TRIALS_FULL/OPTUNA_TRIALS_SMOKE:.0f} Optuna trials)")
    print(f"  On Colab GPU (sklearn uses CPU cores): expect ~20-45 min with n_jobs=-1.")
    print()
    print("SMOKE TEST PASSED. To run the full pipeline, set SMOKE_TEST=False.")
else:
    print("FULL RUN COMPLETED.")